In [1]:
customers_raw = spark.read.parquet("abfss://ecommerce@onelake.dfs.fabric.microsoft.com/ecommerce_lakehouse.Lakehouse/Files/bronze/customers.parquet")
orders_raw = spark.read.parquet("abfss://ecommerce@onelake.dfs.fabric.microsoft.com/ecommerce_lakehouse.Lakehouse/Files/bronze/orders.parquet")
payments_raw = spark.read.parquet("abfss://ecommerce@onelake.dfs.fabric.microsoft.com/ecommerce_lakehouse.Lakehouse/Files/bronze/payments.parquet")
support_raw = spark.read.parquet("abfss://ecommerce@onelake.dfs.fabric.microsoft.com/ecommerce_lakehouse.Lakehouse/Files/bronze/support_tickets.parquet")
web_raw = spark.read.parquet("abfss://ecommerce@onelake.dfs.fabric.microsoft.com/ecommerce_lakehouse.Lakehouse/Files/bronze/web_activities.parquet")


StatementMeta(, 54600772-7503-483c-8d6f-4c8dd99e40c0, 3, Finished, Available, Finished, False)

In [2]:
display(customers_raw)

StatementMeta(, 54600772-7503-483c-8d6f-4c8dd99e40c0, 4, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 39be668a-c476-47ee-bd4f-68c4ceeded64)

### _**SAVE THE DATA INTO DELTA BRONZE FORMAT IN BRONZE TABLE**_

In [3]:
# Save as Bronze Delta Tables
customers_raw.write.format("delta").mode("overwrite").saveAsTable("customers")
orders_raw.write.format("delta").mode("overwrite").saveAsTable("orders")
payments_raw.write.format("delta").mode("overwrite").saveAsTable("payments")
support_raw.write.format("delta").mode("overwrite").saveAsTable("support")
web_raw.write.format("delta").mode("overwrite").saveAsTable("web")

StatementMeta(, 54600772-7503-483c-8d6f-4c8dd99e40c0, 5, Finished, Available, Finished, False)

### _**DATA CLEANING FOR SILVER STAGE**_

In [4]:
display(customers_raw)

StatementMeta(, 54600772-7503-483c-8d6f-4c8dd99e40c0, 6, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 37d52b4f-c5ec-49f4-8761-0a3c2c977da2)

- ## _**clean customer table**_

In [5]:
from pyspark.sql.functions import *
from pyspark.sql.types import *


customers_clean = (
    customers_raw
    .withColumn("email", lower(trim(col("EMAIL"))))
    .withColumn("name", initcap(trim(col("name"))))
    .withColumn("gender", when(lower(col("gender")).isin("f", "female"), "Female")
                          .when(lower(col("gender")).isin("m", "male"), "Male")
                          .otherwise("Other"))
    .withColumn("dob", to_date(regexp_replace(col("dob"), "/", "-")))
    .withColumn("location", initcap(col("location")))
    .dropDuplicates(["customer_id"])
    .dropna(subset=["customer_id", "email"])
)

display(customers_clean.limit(6))

StatementMeta(, 54600772-7503-483c-8d6f-4c8dd99e40c0, 7, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 0ed7aad8-7e89-4e89-bed0-1be8bcc71c4d)

In [10]:
customers_clean.write.format("delta").mode("overwrite").saveAsTable("silver_customers")

StatementMeta(, 54600772-7503-483c-8d6f-4c8dd99e40c0, 12, Finished, Available, Finished, False)

-  ## _**clean orders table**_

In [7]:
%%sql
SELECT*
FROM orders
LIMIT 5

StatementMeta(, 54600772-7503-483c-8d6f-4c8dd99e40c0, 9, Finished, Available, Finished, False)

<Spark SQL result set with 5 rows and 5 fields>

In [8]:
orders = spark.table("orders")
orders_clean = (
    orders
    .withColumn("order_date", 
                when(col("order_date").rlike("^\d{4}/\d{2}/\d{2}$"), to_date(col("order_date"), "yyyy/MM/dd"))
                .when(col("order_date").rlike("^\d{2}-\d{2}-\d{4}$"), to_date(col("order_date"), "dd-MM-yyyy"))
                .when(col("order_date").rlike("^\d{8}$"), to_date(col("order_date"), "yyyyMMdd"))
                .otherwise(to_date(col("order_date"), "yyyy-MM-dd")))
    .withColumn("amount", col("amount").cast(DoubleType()))
    .withColumn("amount", when(col("amount") < 0, None).otherwise(col("amount")))
    .withColumn("status", initcap(col("status")))
    .dropna(subset=["customer_id", "order_date"])
    .dropDuplicates(["order_id"])
)
display(orders_clean.limit(5))


StatementMeta(, 54600772-7503-483c-8d6f-4c8dd99e40c0, 10, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, d325a813-0883-4a57-ae3d-9f213c76b68a)

In [9]:
orders_clean.write.format("delta").mode("overwrite").saveAsTable("silver_orders")

StatementMeta(, 54600772-7503-483c-8d6f-4c8dd99e40c0, 11, Finished, Available, Finished, False)

- ## _**clean payment table**_

In [12]:
%%sql
SELECT*
FROM payments
LIMIT 5

StatementMeta(, 54600772-7503-483c-8d6f-4c8dd99e40c0, 14, Finished, Available, Finished, False)

<Spark SQL result set with 5 rows and 6 fields>

In [13]:
payments = spark.table("payments")
payments_clean = (
    payments
    .withColumn("payment_date", to_date(regexp_replace(col("payment_date"), "/", "-")))
    .withColumn("payment_method", initcap(col("payment_method")))
    .replace({"creditcard": "Credit Card"}, subset=["payment_method"])
    .withColumn("payment_status", initcap(col("payment_status")))
    .withColumn("amount", col("amount").cast(DoubleType()))
    .withColumn("amount", when(col("amount") < 0, None).otherwise(col("amount")))
    .dropna(subset=["customer_id", "payment_date", "amount"])
)
display(payments_clean.limit(5))

payments_clean.write.format("delta").mode("overwrite").saveAsTable("silver_payments")


StatementMeta(, 54600772-7503-483c-8d6f-4c8dd99e40c0, 15, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, b9585699-cf36-4c0d-8a0d-09fb3d29c09e)

- ## _**clean support table**_

In [15]:
%%sql
SELECT*
FROM support
LIMIT 5

StatementMeta(, 54600772-7503-483c-8d6f-4c8dd99e40c0, 17, Finished, Available, Finished, False)

<Spark SQL result set with 5 rows and 5 fields>

In [16]:
support = spark.table("support")
support_clean = (
    support
    .withColumn("ticket_date", to_date(regexp_replace(col("ticket_date"), "/", "-")))
    .withColumn("issue_type", initcap(trim(col("issue_type"))))
    .withColumn("resolution_status", initcap(trim(col("resolution_status"))))
    .replace({"NA": None, "": None}, subset=["issue_type", "resolution_status"])
    .dropDuplicates(["ticket_id"])
    .dropna(subset=["customer_id", "ticket_date"])
)
display(support_clean.limit(5))

support_clean.write.format("delta").mode("overwrite").saveAsTable("silver_support")

StatementMeta(, 54600772-7503-483c-8d6f-4c8dd99e40c0, 18, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, c81b6f1c-2b00-4962-8490-84ebd9bcb93f)

- ## _**clean web table**_

In [18]:
%%sql
SELECT*
FROM web
LIMIT 5 

StatementMeta(, 54600772-7503-483c-8d6f-4c8dd99e40c0, 20, Finished, Available, Finished, False)

<Spark SQL result set with 5 rows and 5 fields>

In [19]:
web = spark.table("web")
web_clean = (
    web
    .withColumn("session_time", to_date(regexp_replace(col("session_time"), "/", "-")))
    .withColumn("page_viewed", lower(col("page_viewed")))
    .withColumn("device_type", initcap(col("device_type")))
    .dropDuplicates(["session_id"])
    .dropna(subset=["customer_id", "session_time", "page_viewed"])
)
display(web_clean.limit(5))

web_clean.write.format("delta").mode("overwrite").saveAsTable("silver_web")

StatementMeta(, 54600772-7503-483c-8d6f-4c8dd99e40c0, 21, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, fc24efd3-0573-42aa-8100-71b9740efa08)

### _**GOLD TABLE FOR AGRREGATION FOR POWER BI REPORTING**_

In [20]:
cust = spark.table("silver_customers").alias("c")
orders = spark.table("silver_orders").alias("o")
payments = spark.table("silver_payments").alias("p")
support = spark.table("silver_support").alias("s")
web = spark.table("silver_web").alias("w")

customer360 = (
    cust
    .join(orders, "customer_id", "left")
    .join(payments, "customer_id", "left")
    .join(support, "customer_id", "left")
    .join(web, "customer_id", "left")
    .select(
        col("c.customer_id"),
        col("c.name"),
        col("c.email"),
        col("c.gender"),
        col("c.dob"),
        col("c.location"),

        col("o.order_id"),
        col("o.order_date"),
        col("o.amount").alias("order_amount"),
        col("o.status").alias("order_status"),

        col("p.payment_method"),
        col("p.payment_status"),
        col("p.amount").alias("payment_amount"),

        col("s.ticket_id"),
        col("s.issue_type"),
        col("s.ticket_date"),
        col("s.resolution_status"),

        col("w.page_viewed"),
        col("w.device_type"),
        col("w.session_time")
    )
)
display(customer360.limit(10))

customer360.write.format("delta").mode("overwrite").saveAsTable("gold_customer360")

StatementMeta(, 54600772-7503-483c-8d6f-4c8dd99e40c0, 22, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, efe0622c-b7ba-40c7-9c79-4db8d3f62d0e)